PINN NETWORK FOR THERMAL ABLATION (TRAINING CODE)

In [ ]:
# ============================================================
# PINN-style Bioheat + Death (ZIP -> extract -> train)
# Minimal publishable upgrades:
#  1) Validation = full-volume RMSE% (subset per epoch)
#  2) Importance sampling (death-focused supervision)  [FIXED GPU/CPU mismatch]
#  3) Loss balancing (adaptive via grad-norms; no torch.optim)
#  4) Variable coefficients (rho,c,perfusion + k(x) field)
# Train for 2mm, 3mm, 4mm
# ============================================================

import os, math, random, zipfile, sys, subprocess, time, shutil
from dataclasses import dataclass
from typing import Tuple, Optional, Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ----------------------------
# 0) Install missing deps (Jupyter-safe)
# ----------------------------
def pip_install(pkg: str):
    print(f"Installing {pkg} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import tifffile as tiff
except ModuleNotFoundError:
    pip_install("tifffile")
    import tifffile as tiff

try:
    import imageio.v3 as iio
except ModuleNotFoundError:
    pip_install("imageio")
    import imageio.v3 as iio


# ----------------------------
# ZIP helper (Windows) - separate train/val ZIPs
# ----------------------------
TRAIN_ZIP_PATH = r"C:\Users\quaid\Desktop\training.zip"
VAL_ZIP_PATH   = r"C:\Users\quaid\Desktop\validation.zip"
EXTRACT_DIR    = r"C:\Users\quaid\Desktop\Dataset_extracted"

def unzip_if_needed(zip_path: str, extract_dir: str) -> str:
    os.makedirs(extract_dir, exist_ok=True)
    if os.path.isdir(extract_dir) and len(os.listdir(extract_dir)) > 0:
        print(f"Extraction folder already exists and is not empty: {extract_dir}. Skipping unzip.")
        return extract_dir

    print(f"Extracting ZIP: {zip_path} -> {extract_dir}")
    t0 = time.time()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print("Extraction done in", round(time.time()-t0,1), "sec")
    return extract_dir

def prepare_dataset_root(train_zip: str, val_zip: str, base_extract_dir: str) -> str:
    os.makedirs(base_extract_dir, exist_ok=True)

    train_dir = os.path.join(base_extract_dir, "training")
    val_dir   = os.path.join(base_extract_dir, "validation")

    if os.path.isdir(train_dir) and len(os.listdir(train_dir)) > 0 and \
       os.path.isdir(val_dir) and len(os.listdir(val_dir)) > 0:
        print("Prepared dataset root already exists. Skipping extraction.")
        return base_extract_dir

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    print("Extracting training ZIP...")
    t0 = time.time()
    with zipfile.ZipFile(train_zip, 'r') as zf:
        zf.extractall(train_dir)
    print("Training extracted in", round(time.time()-t0,1), "sec")

    print("Extracting validation ZIP...")
    t0 = time.time()
    with zipfile.ZipFile(val_zip, 'r') as zf:
        zf.extractall(val_dir)
    print("Validation extracted in", round(time.time()-t0,1), "sec")

    # Flatten one extra nested folder if present
    def flatten_if_needed(split_dir: str):
        items = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
        files = [f for f in os.listdir(split_dir) if os.path.isfile(os.path.join(split_dir, f))]
        if len(items) == 1 and len(files) == 0:
            nested = os.path.join(split_dir, items[0])
            nested_items = os.listdir(nested)
            for name in nested_items:
                shutil.move(os.path.join(nested, name), os.path.join(split_dir, name))
            os.rmdir(nested)

    flatten_if_needed(train_dir)
    flatten_if_needed(val_dir)

    return base_extract_dir

def find_dataset_root(search_dir: str) -> str:
    search_dir = os.path.normpath(search_dir)

    def has_splits(p: str) -> bool:
        return os.path.isdir(os.path.join(p, "training")) and os.path.isdir(os.path.join(p, "validation"))

    if has_splits(search_dir):
        return search_dir

    try:
        kids = [d for d in os.listdir(search_dir) if os.path.isdir(os.path.join(search_dir, d))]
        if len(kids) == 1:
            cand = os.path.join(search_dir, kids[0])
            if has_splits(cand):
                return cand
    except:
        pass

    for path, dirs, _ in os.walk(search_dir):
        if "training" in dirs and "validation" in dirs:
            return path

    raise FileNotFoundError(f"Could not find dataset root containing training/ and validation/ inside: {search_dir}")


# ----------------------------
# 1) Config
# ----------------------------
@dataclass
class CFG:
    root: str
    split_train: str = "training"
    split_val: str = "validation"

    resolutions: Tuple[str, ...] = ("2mm","3mm","4mm")

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

    channel_tags: Tuple[str, ...] = ("applicator","perfusion","density","specific_heat","slice","death")

    batch_volumes: int = 1
    n_data_pts: int = 4096
    n_collocation: int = 4096
    n_ic_pts: int = 2048

    t_final: float = 1.0
    lr: float = 1e-4
    epochs: int = 20

    Tb: float = 0.0

    k0: float = 0.02
    k1: float = 0.10
    q0: float = 1.0

    w_data: float = 1.0
    w_pde: float = 1.0
    w_ic: float = 0.2

    focus_frac: float = 0.60
    focus_topk: float = 0.20

    val_subset: int = 10

    beta1: float = 0.9
    beta2: float = 0.999
    eps: float = 1e-8

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ----------------------------
# 2) Dataset loader (TIFF/PNG volumes)
# ----------------------------
def load_vol(path: str) -> np.ndarray:
    p = path.lower()
    if p.endswith((".tif",".tiff")):
        arr = tiff.imread(path).astype(np.float32)
        if arr.ndim == 2:
            arr = arr[None,:,:]
        return arr
    if p.endswith(".png"):
        arr = iio.imread(path).astype(np.float32)
        if arr.ndim == 3:
            arr = arr[...,0]
        if arr.max() > 1.5:
            arr /= 255.0
        arr = arr[None,:,:]
        return arr
    raise ValueError(f"Unsupported file: {path}")

class AblationVoxelDataset(Dataset):
    def __init__(self, root: str, split: str, resolution: str, tags: Tuple[str,...]):
        self.split_path = os.path.join(root, split)
        self.res = resolution
        self.tags = tags

        if not os.path.isdir(self.split_path):
            raise FileNotFoundError(f"Split path not found: {self.split_path}")

        syn_folders = sorted(
            [d for d in os.listdir(self.split_path) if d.startswith("synthetic_")],
            key=lambda x:int(x.split("_")[1])
        )

        self.samples = []
        for syn in syn_folders:
            res_dir = os.path.join(self.split_path, syn, resolution)
            if not os.path.isdir(res_dir):
                continue

            groups: Dict[Tuple[int,int], Dict[str,str]] = {}
            for fn in os.listdir(res_dir):
                if not fn.lower().endswith((".tif",".tiff",".png")):
                    continue
                base = os.path.splitext(fn)[0]
                parts = base.split("_")
                if len(parts) < 4 or parts[0] != "synthetic":
                    continue
                try:
                    i = int(parts[1]); j = int(parts[2])
                except:
                    continue
                tag = "_".join(parts[3:])
                if tag not in tags:
                    continue
                groups.setdefault((i,j), {})[tag] = os.path.join(res_dir, fn)

            for _, mp in groups.items():
                if all(t in mp for t in tags):
                    self.samples.append(mp)

        if len(self.samples) == 0:
            raise RuntimeError(f"No samples found in {self.split_path}/{resolution}. Check naming/extensions.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mp = self.samples[idx]
        vols = {t: load_vol(mp[t]) for t in self.tags}

        Dref = vols["applicator"].shape[0]
        if vols["slice"].shape[0] != Dref:
            if vols["slice"].shape[0] == 1:
                vols["slice"] = np.repeat(vols["slice"], Dref, axis=0)
            else:
                vols["slice"] = vols["slice"][:Dref]

        x = np.stack([vols[t] for t in self.tags], axis=0).astype(np.float32)
        return torch.from_numpy(x)


# ----------------------------
# 3) PINN network
# ----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=128, depth=6):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class BioheatPINN(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.mlp = MLP(in_dim, out_dim=2, width=128, depth=6)
        self.log_k_scale = nn.Parameter(torch.tensor(0.0))
        self.log_perf_scale = nn.Parameter(torch.tensor(0.0))

    def forward(self, inp):
        out = self.mlp(inp)
        T = out[:, 0:1]
        D = torch.sigmoid(out[:, 1:2])
        return T, D

    def k_multiplier(self):
        return torch.exp(self.log_k_scale)

    def perf_multiplier(self):
        return torch.exp(self.log_perf_scale)


# ----------------------------
# 4) Manual Adam Optimizer (NO torch.optim)
# ----------------------------
class ManualAdam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr
        self.b1 = beta1
        self.b2 = beta2
        self.eps = eps
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    @torch.no_grad()
    def step(self):
        self.t += 1
        b1, b2 = self.b1, self.b2
        lr = self.lr

        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad
            self.m[i].mul_(b1).add_(g, alpha=1 - b1)
            self.v[i].mul_(b2).addcmul_(g, g, value=1 - b2)
            mhat = self.m[i] / (1 - b1 ** self.t)
            vhat = self.v[i] / (1 - b2 ** self.t)
            p.addcdiv_(mhat, vhat.sqrt().add_(self.eps), value=-lr)

    @torch.no_grad()
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


# ----------------------------
# 5) Sampling utilities (importance sampling) [FIXED DEVICE]
# ----------------------------
def sample_points_from_volume_uniform(vol: torch.Tensor, cfg: CFG, n_pts: int):
    device = cfg.device
    C,D,H,W = vol.shape

    zz = torch.randint(0, D, (n_pts,), device=device)
    yy = torch.randint(0, H, (n_pts,), device=device)
    xx = torch.randint(0, W, (n_pts,), device=device)

    x = xx.float() / max(W-1, 1)
    y = yy.float() / max(H-1, 1)
    z = zz.float() / max(D-1, 1)

    tag_to_idx = {t:i for i,t in enumerate(cfg.channel_tags)}
    appl = vol[tag_to_idx["applicator"], zz, yy, xx]
    perf = vol[tag_to_idx["perfusion"], zz, yy, xx]
    rho  = vol[tag_to_idx["density"], zz, yy, xx]
    c    = vol[tag_to_idx["specific_heat"], zz, yy, xx]
    slc  = vol[tag_to_idx["slice"], zz, yy, xx]
    death= vol[tag_to_idx["death"], zz, yy, xx]

    params = torch.stack([appl, perf, rho, c, slc], dim=1)
    coords_xyz = torch.stack([x,y,z], dim=1)
    return coords_xyz, params, death

def sample_points_from_volume_importance(vol: torch.Tensor, cfg: CFG, n_pts: int):
    device = vol.device
    C,D,H,W = vol.shape
    tag_to_idx = {t:i for i,t in enumerate(cfg.channel_tags)}
    death_vol = vol[tag_to_idx["death"]].reshape(-1)

    n_focus = int(round(cfg.focus_frac * n_pts))
    n_uni = n_pts - n_focus

    k = max(1, int(cfg.focus_topk * death_vol.numel()))
    _, top_idx = torch.topk(death_vol, k=k, largest=True, sorted=False)

    focus_choice = top_idx[torch.randint(0, top_idx.numel(), (n_focus,), device=device)]
    uni_choice = torch.randint(0, death_vol.numel(), (n_uni,), device=device)

    all_choice = torch.cat([focus_choice, uni_choice], dim=0)

    zz = all_choice // (H*W)
    rem = all_choice % (H*W)
    yy = rem // W
    xx = rem % W

    x = xx.float() / max(W-1, 1)
    y = yy.float() / max(H-1, 1)
    z = zz.float() / max(D-1, 1)

    appl = vol[tag_to_idx["applicator"], zz, yy, xx]
    perf = vol[tag_to_idx["perfusion"], zz, yy, xx]
    rho  = vol[tag_to_idx["density"], zz, yy, xx]
    c    = vol[tag_to_idx["specific_heat"], zz, yy, xx]
    slc  = vol[tag_to_idx["slice"], zz, yy, xx]
    death= vol[tag_to_idx["death"], zz, yy, xx]

    params = torch.stack([appl, perf, rho, c, slc], dim=1)
    coords_xyz = torch.stack([x,y,z], dim=1)
    return coords_xyz, params, death


# ----------------------------
# 6) Variable-coefficient bioheat residual
# ----------------------------
def bioheat_residual_varcoeff(T, coords_xyz, t, params, model: BioheatPINN, cfg: CFG):
    appl = params[:,0:1]
    perf = params[:,1:2]
    rho  = params[:,2:3]
    c    = params[:,3:4]
    slc  = params[:,4:5]

    kx = (cfg.k0 + cfg.k1 * torch.sigmoid(10.0*(slc - 0.5))) * model.k_multiplier()

    T_t = torch.autograd.grad(T, t, grad_outputs=torch.ones_like(T),
                              create_graph=True, retain_graph=True)[0]

    grads = torch.autograd.grad(T, coords_xyz, grad_outputs=torch.ones_like(T),
                                create_graph=True, retain_graph=True)[0]
    Tx, Ty, Tz = grads[:,0:1], grads[:,1:2], grads[:,2:3]

    kTx = kx * Tx
    kTy = kx * Ty
    kTz = kx * Tz

    d_kTx = torch.autograd.grad(kTx, coords_xyz, grad_outputs=torch.ones_like(kTx),
                                create_graph=True, retain_graph=True)[0][:,0:1]
    d_kTy = torch.autograd.grad(kTy, coords_xyz, grad_outputs=torch.ones_like(kTy),
                                create_graph=True, retain_graph=True)[0][:,1:2]
    d_kTz = torch.autograd.grad(kTz, coords_xyz, grad_outputs=torch.ones_like(kTz),
                                create_graph=True, retain_graph=True)[0][:,2:3]
    div_k_grad = d_kTx + d_kTy + d_kTz

    Qext = cfg.q0 * appl
    Qbio = model.perf_multiplier() * perf * (cfg.Tb - T)

    return rho*c*T_t - div_k_grad - Qbio - Qext


# ----------------------------
# 7) Metrics: full-volume RMSE%
# ----------------------------
@torch.no_grad()
def predict_death_volume(model: BioheatPINN, vol: torch.Tensor, cfg: CFG, t_final: float, chunk: int = 150000):
    model.eval()
    device = cfg.device
    vol = vol.to(device)
    C,D,H,W = vol.shape
    idx = {t:i for i,t in enumerate(cfg.channel_tags)}

    appl = vol[idx["applicator"]].reshape(-1)
    perf = vol[idx["perfusion"]].reshape(-1)
    rho  = vol[idx["density"]].reshape(-1)
    c    = vol[idx["specific_heat"]].reshape(-1)
    slc  = vol[idx["slice"]].reshape(-1)

    zz, yy, xx = torch.meshgrid(torch.arange(D, device=device),
                                torch.arange(H, device=device),
                                torch.arange(W, device=device), indexing="ij")
    x = (xx.reshape(-1).float() / max(W-1,1))
    y = (yy.reshape(-1).float() / max(H-1,1))
    z = (zz.reshape(-1).float() / max(D-1,1))
    t = torch.full_like(x, float(t_final))

    params = torch.stack([appl, perf, rho, c, slc], dim=1)

    out = torch.empty((D*H*W,), device=device, dtype=torch.float32)
    N = out.numel()
    for s in range(0, N, chunk):
        e = min(N, s+chunk)
        inp = torch.cat([x[s:e,None], y[s:e,None], z[s:e,None], t[s:e,None], params[s:e]], dim=1)
        _, Dpred = model(inp)
        out[s:e] = Dpred.squeeze(1).float()

    return out.view(D,H,W).detach().cpu().numpy()

def rmse_percent(pred: np.ndarray, target: np.ndarray, eps=1e-8) -> float:
    pred = np.clip(pred, 0.0, 1.0)
    mse = np.mean((pred - target) ** 2)
    return 100.0 * math.sqrt(mse + eps)


# ----------------------------
# 8) Loss balancing (adaptive via grad norms)
# ----------------------------
def grad_norm(loss, params):
    grads = torch.autograd.grad(loss, params, retain_graph=True, create_graph=False, allow_unused=True)
    s = 0.0
    for g in grads:
        if g is None:
            continue
        s = s + (g.detach().pow(2).sum())
    return torch.sqrt(s + 1e-12)

def compute_adaptive_weights(loss_data, loss_pde, loss_ic, model, cfg: CFG):
    params = [p for p in model.parameters() if p.requires_grad]
    g_data = grad_norm(loss_data, params)
    g_pde  = grad_norm(loss_pde,  params)
    g_ic   = grad_norm(loss_ic,   params)

    g_ref = (g_data + g_pde + g_ic) / 3.0

    w_data = float(cfg.w_data) * float((g_ref / (g_data + 1e-12)).clamp(0.2, 5.0))
    w_pde  = float(cfg.w_pde)  * float((g_ref / (g_pde  + 1e-12)).clamp(0.2, 5.0))
    w_ic   = float(cfg.w_ic)   * float((g_ref / (g_ic   + 1e-12)).clamp(0.2, 5.0))
    return w_data, w_pde, w_ic


# ----------------------------
# 9) Train one resolution
# ----------------------------
def train_one_resolution(cfg: CFG, resolution: str):
    set_seed(cfg.seed)

    train_ds = AblationVoxelDataset(cfg.root, cfg.split_train, resolution, cfg.channel_tags)
    val_ds   = AblationVoxelDataset(cfg.root, cfg.split_val,   resolution, cfg.channel_tags)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_volumes, shuffle=True, num_workers=0)

    model = BioheatPINN(in_dim=4+5).to(cfg.device)
    opt = ManualAdam(model.parameters(), lr=cfg.lr, beta1=cfg.beta1, beta2=cfg.beta2, eps=cfg.eps)

    print(f"\n==================== RESOLUTION {resolution} ====================")
    print("Device:", cfg.device, "| Train vols:", len(train_ds), "| Val vols:", len(val_ds))

    val_indices = list(range(len(val_ds)))
    random.shuffle(val_indices)
    val_indices = val_indices[: min(cfg.val_subset, len(val_indices))]

    best_val = float("inf")
    best_state = None

    for ep in range(1, cfg.epochs+1):
        model.train()
        running = []
        t_ep = time.time()

        for vol_batch in train_loader:
            vol = vol_batch[0].to(cfg.device)

            xyz_d, params_d, death_gt = sample_points_from_volume_importance(vol, cfg, cfg.n_data_pts)
            t_d = torch.full((xyz_d.shape[0],1), cfg.t_final, device=cfg.device)
            inp_d = torch.cat([xyz_d, t_d, params_d], dim=1)
            _, D_pred = model(inp_d)
            loss_data = F.mse_loss(D_pred, death_gt.unsqueeze(1))

            xyz_c, params_c, _ = sample_points_from_volume_uniform(vol, cfg, cfg.n_collocation)
            t_c = torch.rand((xyz_c.shape[0],1), device=cfg.device) * cfg.t_final
            xyz_c.requires_grad_(True)
            t_c.requires_grad_(True)
            inp_c = torch.cat([xyz_c, t_c, params_c], dim=1)
            T_c, _ = model(inp_c)
            res = bioheat_residual_varcoeff(T_c, xyz_c, t_c, params_c, model, cfg)
            loss_pde = torch.mean(res**2)

            xyz0, params0, _ = sample_points_from_volume_uniform(vol, cfg, cfg.n_ic_pts)
            t0 = torch.zeros((xyz0.shape[0],1), device=cfg.device)
            inp0 = torch.cat([xyz0, t0, params0], dim=1)
            T0, D0 = model(inp0)
            loss_ic = F.mse_loss(T0, torch.full_like(T0, cfg.Tb)) + F.mse_loss(D0, torch.zeros_like(D0))

            w_data, w_pde, w_ic = compute_adaptive_weights(loss_data, loss_pde, loss_ic, model, cfg)
            loss = w_data*loss_data + w_pde*loss_pde + w_ic*loss_ic

            opt.zero_grad()
            loss.backward()
            opt.step()

            running.append([loss.item(), loss_data.item(), loss_pde.item(), loss_ic.item(), w_data, w_pde, w_ic])

        arr = np.mean(running, axis=0)
        print(f"Epoch {ep:03d} | total={arr[0]:.4e} data={arr[1]:.4e} pde={arr[2]:.4e} ic={arr[3]:.4e} "
              f"| w=({arr[4]:.2f},{arr[5]:.2f},{arr[6]:.2f}) | time={time.time()-t_ep:.1f}s")

        model.eval()
        rmses = []
        with torch.no_grad():
            for vi in val_indices:
                vvol = val_ds[vi].to(cfg.device)
                target = vvol[cfg.channel_tags.index("death")].detach().cpu().numpy()
                pred = predict_death_volume(model, vvol, cfg, t_final=cfg.t_final, chunk=150000)
                rmses.append(rmse_percent(pred, target))
        val_rmse = float(np.mean(rmses))
        val_std  = float(np.std(rmses))
        print(f"   ✅ Val full-volume RMSE% (subset n={len(val_indices)}): {val_rmse:.3f} ± {val_std:.3f}")

        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            print("   ⭐ New best model saved (in memory).")

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val


# ----------------------------
# 10) Run all resolutions
# ----------------------------
prepared_root = prepare_dataset_root(TRAIN_ZIP_PATH, VAL_ZIP_PATH, EXTRACT_DIR)
dataset_root = find_dataset_root(prepared_root)
print("✅ Using dataset root:", dataset_root)

cfg = CFG(root=dataset_root)
print("CUDA available:", torch.cuda.is_available(), "| Device:", cfg.device)

models = {}
scores = {}

for res in cfg.resolutions:
    m, best = train_one_resolution(cfg, res)
    models[res] = m
    scores[res] = best

print("\n==================== SUMMARY ====================")
for r in cfg.resolutions:
    print(f"{r}: best val RMSE% = {scores[r]:.3f}")

PINN NETWORK FOR THERMAL ABLATION (TESTING CODE)

In [ ]:
# ============================================================
# PINN Testing + Visualization (C-NCA style figures) - FIXED
# + ADDED: figure saving (PNG + PDF) to a folder
# + CHANGED: titles ONLY "Target" and "Output" (no RMSE text on figure)
# + UPDATED: testing zip path to: C:\Users\quaid\Desktop\testing.zip
#
# TEST SET ONLY (ZIP -> extract -> evaluate -> plots -> save)
# ============================================================

import os, math, random, zipfile, time, sys, subprocess
from typing import Tuple, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


# ----------------------------
# 0) Install deps (Jupyter-safe)
# ----------------------------
def pip_install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import tifffile as tiff
except ModuleNotFoundError:
    pip_install("tifffile")
    import tifffile as tiff

try:
    import imageio.v3 as iio
except ModuleNotFoundError:
    pip_install("imageio")
    import imageio.v3 as iio


# ----------------------------
# 1) SAFE CUDA SYNC (prevents your crash)
# ----------------------------
def safe_cuda_sync():
    if torch.cuda.is_available() and hasattr(torch, "device"):
        try:
            torch.cuda.synchronize()
        except Exception:
            pass


# ----------------------------
# 2) Test ZIP extraction (ONLY test link)
# ----------------------------
TEST_ZIP_PATH = r"C:\Users\quaid\Desktop\testing.zip"
TEST_EXTRACT_DIR = r"C:\Users\quaid\Desktop\testing_extracted"

def unzip_if_needed(zip_path: str, extract_dir: str) -> str:
    os.makedirs(extract_dir, exist_ok=True)
    if os.path.isdir(extract_dir) and len(os.listdir(extract_dir)) > 0:
        print("✅ Test already extracted:", extract_dir)
        return extract_dir
    print("Unzipping test set...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print("✅ Unzip done in", round(time.time()-t0, 1), "sec")
    return extract_dir

def find_testing_dir(search_dir: str) -> str:
    search_dir = os.path.normpath(search_dir)

    cand = os.path.join(search_dir, "testing")
    if os.path.isdir(cand):
        kids = [d for d in os.listdir(cand) if d.startswith("synthetic_")]
        if len(kids) > 0:
            return cand

    try:
        kids = [d for d in os.listdir(search_dir) if d.startswith("synthetic_")]
        if len(kids) > 0:
            return search_dir
    except:
        pass

    for path, dirs, _ in os.walk(search_dir):
        syns = [d for d in dirs if d.startswith("synthetic_")]
        if len(syns) > 0:
            return path

    raise FileNotFoundError(f"Could not locate testing directory containing synthetic_* inside: {search_dir}")


# ----------------------------
# 3) Device info
# ----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available(), "| Device:", DEVICE)
print("torch has torch.device:", hasattr(torch, "device"))
if torch.cuda.is_available() and hasattr(torch, "device"):
    try:
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU mem (GB):", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
    except Exception:
        pass


# ----------------------------
# 4) Dataset reading (TIFF/PNG)
# ----------------------------
CHANNEL_TAGS = ("applicator","perfusion","density","specific_heat","slice","death")
RESOLUTIONS = ("2mm","3mm","4mm")

def load_vol(path: str) -> np.ndarray:
    p = path.lower()
    if p.endswith((".tif",".tiff")):
        arr = tiff.imread(path).astype(np.float32)
        if arr.ndim == 2:
            arr = arr[None,:,:]
        return arr
    if p.endswith(".png"):
        arr = iio.imread(path).astype(np.float32)
        if arr.ndim == 3:
            arr = arr[...,0]
        if arr.max() > 1.5:
            arr /= 255.0
        arr = arr[None,:,:]
        return arr
    raise ValueError(f"Unsupported file: {path}")

def parse_name(filename: str):
    base = os.path.splitext(filename)[0]
    parts = base.split("_")
    if len(parts) < 4 or parts[0] != "synthetic":
        return None
    try:
        i = int(parts[1]); j = int(parts[2])
    except:
        return None
    tag = "_".join(parts[3:])
    return i, j, tag

class TestVoxelDataset(torch.utils.data.Dataset):
    def __init__(self, testing_dir: str, resolution: str, tags: Tuple[str,...]=CHANNEL_TAGS):
        self.testing_dir = testing_dir
        self.resolution = resolution
        self.tags = tags

        syn_folders = sorted([d for d in os.listdir(testing_dir) if d.startswith("synthetic_")],
                             key=lambda x:int(x.split("_")[1]))

        self.samples: List[Dict[str,str]] = []
        for syn in syn_folders:
            res_dir = os.path.join(testing_dir, syn, resolution)
            if not os.path.isdir(res_dir):
                continue

            groups: Dict[Tuple[int,int], Dict[str,str]] = {}
            for fn in os.listdir(res_dir):
                if not fn.lower().endswith((".tif",".tiff",".png")):
                    continue
                parsed = parse_name(fn)
                if parsed is None:
                    continue
                i, j, tag = parsed
                if tag not in tags:
                    continue
                groups.setdefault((i,j), {})[tag] = os.path.join(res_dir, fn)

            for _, mp in groups.items():
                if all(t in mp for t in tags):
                    self.samples.append(mp)

        if len(self.samples) == 0:
            dbg = os.path.join(testing_dir, "synthetic_0", resolution)
            raise RuntimeError(f"No test samples found for {resolution}. Debug folder: {dbg}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        mp = self.samples[idx]
        vols = {t: load_vol(mp[t]) for t in self.tags}

        Dref = vols["applicator"].shape[0]
        if vols["slice"].shape[0] != Dref:
            if vols["slice"].shape[0] == 1:
                vols["slice"] = np.repeat(vols["slice"], Dref, axis=0)
            else:
                vols["slice"] = vols["slice"][:Dref]

        x = np.stack([vols[t] for t in self.tags], axis=0).astype(np.float32)
        return torch.from_numpy(x)


# ----------------------------
# 5) Model definition (must match your training)
# ----------------------------
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=128, depth=6):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth-1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class BioheatPINN(nn.Module):
    def __init__(self, in_dim=9):
        super().__init__()
        self.mlp = MLP(in_dim, out_dim=2, width=128, depth=6)
        self.log_k_scale = nn.Parameter(torch.tensor(0.0))
        self.log_perf_scale = nn.Parameter(torch.tensor(0.0))
    def forward(self, inp):
        out = self.mlp(inp)
        T = out[:, 0:1]
        D = torch.sigmoid(out[:, 1:2])
        return T, D


# ----------------------------
# 6) Models source: use your in-memory models dict if present
# ----------------------------
if ("models" in globals()) and isinstance(globals().get("models"), dict) and all(r in globals()["models"] for r in RESOLUTIONS):
    MODELS = globals()["models"]
    print("✅ Using trained MODELS from memory.")
else:
    raise RuntimeError(
        "I couldn't find your trained models dict in memory.\n"
        "Run your training cell first so you have: models['2mm'], models['3mm'], models['4mm']."
    )

for r in RESOLUTIONS:
    MODELS[r] = MODELS[r].to(DEVICE).eval()


# ----------------------------
# 7) Full-volume inference + RMSE% + timing
# ----------------------------
def rmse_percent(pred: np.ndarray, target: np.ndarray, eps=1e-8) -> float:
    pred = np.clip(pred, 0.0, 1.0)
    mse = np.mean((pred - target) ** 2)
    return 100.0 * math.sqrt(mse + eps)

@torch.no_grad()
def predict_death_volume(model: BioheatPINN, vol: torch.Tensor, t_final: float = 1.0, chunk: int = 120000):
    model.eval()
    vol = vol.to(DEVICE, non_blocking=True)

    idx = {t:i for i,t in enumerate(CHANNEL_TAGS)}
    C,D,H,W = vol.shape

    appl = vol[idx["applicator"]].reshape(-1)
    perf = vol[idx["perfusion"]].reshape(-1)
    rho  = vol[idx["density"]].reshape(-1)
    c    = vol[idx["specific_heat"]].reshape(-1)
    slc  = vol[idx["slice"]].reshape(-1)

    zz, yy, xx = torch.meshgrid(torch.arange(D, device=DEVICE),
                                torch.arange(H, device=DEVICE),
                                torch.arange(W, device=DEVICE), indexing="ij")
    x = (xx.reshape(-1).float() / max(W-1,1))
    y = (yy.reshape(-1).float() / max(H-1,1))
    z = (zz.reshape(-1).float() / max(D-1,1))
    t = torch.full_like(x, float(t_final))

    params = torch.stack([appl, perf, rho, c, slc], dim=1)

    out = torch.empty((D*H*W,), device=DEVICE, dtype=torch.float32)
    N = out.numel()
    for s in range(0, N, chunk):
        e = min(N, s+chunk)
        inp = torch.cat([x[s:e,None], y[s:e,None], z[s:e,None], t[s:e,None], params[s:e]], dim=1)
        _, Dpred = model(inp)
        out[s:e] = Dpred.squeeze(1).float()

    return out.view(D,H,W).detach().cpu().numpy()

def eval_resolution(testing_dir: str, resolution: str, model: BioheatPINN, max_cases: Optional[int] = None):
    ds = TestVoxelDataset(testing_dir, resolution, CHANNEL_TAGS)
    n = len(ds) if max_cases is None else min(len(ds), max_cases)

    rmses, times = [], []
    for i in range(n):
        vol = ds[i]
        target = vol[CHANNEL_TAGS.index("death")].numpy()

        safe_cuda_sync()
        t0 = time.time()
        pred = predict_death_volume(model, vol, t_final=1.0, chunk=120000)
        safe_cuda_sync()
        dt = time.time() - t0

        rmses.append(rmse_percent(pred, target))
        times.append(dt)

    rmses = np.array(rmses, dtype=float)
    times = np.array(times, dtype=float)

    return {
        "n": n,
        "rmse_mean": float(rmses.mean()),
        "rmse_std": float(rmses.std()),
        "time_mean": float(times.mean()),
        "time_std": float(times.std()),
        "fps_mean": float(1.0 / max(times.mean(), 1e-9)),
        "rmse_all": rmses,
        "time_all": times,
        "dataset": ds,
    }


# ----------------------------
# 8) Fig.3-style overlay visualization (Target vs Output)
# + ADDED: saving
# + FIXED: highly uniform, symmetric panels, no unusual white spacing
# ----------------------------
SAVE_DIR = r"C:\Users\quaid\Desktop\pinn_test_figures"
os.makedirs(SAVE_DIR, exist_ok=True)
print("✅ Figures will be saved to:", SAVE_DIR)

def savefig(fig, name_base: str, dpi: int = 400):
    png_path = os.path.join(SAVE_DIR, name_base + ".png")
    pdf_path = os.path.join(SAVE_DIR, name_base + ".pdf")
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", pad_inches=0.01, facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", pad_inches=0.01, facecolor="white")
    print("Saved:", png_path)
    plt.close(fig)

def make_overlay(death: np.ndarray, vessels_like: np.ndarray, applicator: np.ndarray):
    death = np.clip(death, 0, 1)
    vessels_like = np.clip(vessels_like, 0, 1)
    applicator = np.clip(applicator, 0, 1)

    rgb = np.zeros((death.shape[0], death.shape[1], 3), dtype=np.float32)
    rgb[..., 0] = np.clip(death + applicator, 0, 1)  # red
    rgb[..., 1] = applicator                         # green -> makes applicator yellow
    rgb[..., 2] = vessels_like                      # blue (proxy)
    return rgb

def show_target_vs_output(
    vol: torch.Tensor,
    pred_death: np.ndarray,
    z_slice: Optional[int] = None,
    title_suffix: str = "",
    save_name: Optional[str] = None,
):
    idx = {t:i for i,t in enumerate(CHANNEL_TAGS)}
    death_gt = vol[idx["death"]].numpy()
    appl = vol[idx["applicator"]].numpy()
    slc  = vol[idx["slice"]].numpy()

    D = death_gt.shape[0]
    if z_slice is None:
        z_slice = D // 2

    tgt_rgb = make_overlay(death_gt[z_slice], slc[z_slice], appl[z_slice])
    out_rgb = make_overlay(pred_death[z_slice], slc[z_slice], appl[z_slice])

    # Use square-ish and exactly matched axes for both panels
    fig = plt.figure(figsize=(8.0, 4.0), dpi=250, facecolor="white")
    gs = fig.add_gridspec(
        1, 2,
        left=0.02, right=0.98,
        bottom=0.04, top=0.88,
        wspace=0.02
    )

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])

    for a, img, ttl in zip([ax1, ax2], [tgt_rgb, out_rgb], ["Target" + title_suffix, "Output" + title_suffix]):
        a.imshow(img, interpolation="nearest", aspect="equal")
        a.set_title(ttl, pad=4)
        a.set_xticks([])
        a.set_yticks([])
        a.set_box_aspect(1)
        a.set_anchor("C")
        for spine in a.spines.values():
            spine.set_visible(False)

    if save_name is not None:
        savefig(fig, save_name)
    else:
        plt.show()


# ----------------------------
# 9) RUN testing + plots (and save)
# ----------------------------
extracted = unzip_if_needed(TEST_ZIP_PATH, TEST_EXTRACT_DIR)
testing_dir = find_testing_dir(extracted)
print("✅ testing_dir:", testing_dir)

results = {}
for res in RESOLUTIONS:
    print(f"\n=== Evaluating {res} ===")
    results[res] = eval_resolution(testing_dir, res, MODELS[res], max_cases=None)
    r = results[res]
    print(f"n={r['n']} | RMSE%={r['rmse_mean']:.3f} ± {r['rmse_std']:.3f} "
          f"| time={r['time_mean']:.4f}s ± {r['time_std']:.4f}s | fps≈{r['fps_mean']:.1f}")

print("\n==================== SUMMARY (PINN) ====================")
print("Resolution | Avg time (s) | STD (s) | FPS | Avg RMSE% | STD RMSE%")
for res in RESOLUTIONS:
    r = results[res]
    print(f"{res:>9} | {r['time_mean']:.4f}      | {r['time_std']:.4f}  | {r['fps_mean']:.1f} | {r['rmse_mean']:.3f}   | {r['rmse_std']:.3f}")

# ---- Save violin plot ----
rmse_lists = [results[r]["rmse_all"] for r in RESOLUTIONS]
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4), dpi=250, facecolor="white")
ax.violinplot(rmse_lists, showmeans=True, showmedians=True)
ax.set_xticks([1,2,3]); ax.set_xticklabels(["2mm","3mm","4mm"])
ax.set_ylabel("RMSE (%)")
ax.set_title("PINN RMSE distribution vs voxel size (testing)")
ax.grid(True, alpha=0.25)
fig.subplots_adjust(left=0.12, right=0.97, bottom=0.14, top=0.90)
savefig(fig, "Fig_RMSE_violin_vs_voxel")

# ---- Save FPS plot ----
fps = [results[r]["fps_mean"] for r in RESOLUTIONS]
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4), dpi=250, facecolor="white")
ax.plot([2,3,4], fps, marker="o")
ax.set_xlabel("Voxel size (mm)")
ax.set_ylabel("Frequency (Hz) ~ volumes/sec")
ax.set_title("PINN inference frequency vs voxel size (testing)")
ax.grid(True, alpha=0.25)
fig.subplots_adjust(left=0.12, right=0.97, bottom=0.14, top=0.90)
savefig(fig, "Fig_FPS_vs_voxel")

# ---- Qualitative examples (4 random 2mm cases) + save ----
print("\n==================== QUALITATIVE (Fig.3-style) ====================")
RES_FOR_FIG3 = "2mm"
ds2 = results[RES_FOR_FIG3]["dataset"]

random.seed(0)
idxs = random.sample(range(len(ds2)), k=min(4, len(ds2)))

for k, i in enumerate(idxs, 1):
    vol = ds2[i]
    safe_cuda_sync()
    t0 = time.time()
    pred = predict_death_volume(MODELS[RES_FOR_FIG3], vol, t_final=1.0, chunk=120000)
    safe_cuda_sync()
    dt = time.time() - t0
    print(f"Case {k}: inference time={dt:.3f}s")

    save_name = f"Fig3_style_{RES_FOR_FIG3}_case{k:02d}"
    show_target_vs_output(vol, pred, z_slice=None, title_suffix=f"  ({RES_FOR_FIG3})", save_name=save_name)

print("\n✅ Done. All figures saved in:", SAVE_DIR)